## Check GPU

In [1]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU (T4)')

ModuleNotFoundError: No module named 'torch'

## Mount Google Drive

In [81]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit these paths ────────────────────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/clarity-project/pyq_dataset.jsonl'
OUTPUT_DIR   = '/content/drive/MyDrive/clarity-project/outputs/t5_large_exam'
LOG_DIR      = '/content/drive/MyDrive/clarity-project/outputs/logs'
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print('Drive mounted.')
print(f'Dataset : {DATASET_PATH}')
print(f'Outputs : {OUTPUT_DIR}')

# Verify dataset exists
if os.path.exists(DATASET_PATH):
    size = os.path.getsize(DATASET_PATH) / 1024
    print(f'✅ Dataset found ({size:.1f} KB)')
else:
    print('❌ Dataset NOT found — upload pyq_dataset.jsonl to Google Drive first')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
Dataset : /content/drive/MyDrive/clarity-project/pyq_dataset.jsonl
Outputs : /content/drive/MyDrive/clarity-project/outputs/t5_large_exam
✅ Dataset found (27533.2 KB)


## Install Dependencies

In [82]:
!pip install -q \
    'transformers==4.44.2' \
    'sentencepiece>=0.1.99' \
    'tokenizers>=0.19.0,<0.20.0' \
    'huggingface-hub>=0.24.0,<0.25.0' \
    'accelerate>=0.27.0' \
    'safetensors>=0.4.3' \
    'tqdm>=4.66.0' \
    'tensorboard>=2.15.0' \
    'bitsandbytes>=0.41.1' # Added for 8-bit quantization

print('✅ Dependencies updated with bitsandbytes')

✅ Dependencies updated with bitsandbytes




```
# This is formatted as code
```

##  Write `config.py`

In [101]:
config_lines = [
    "from dataclasses import dataclass",
    "from typing import Optional",
    "",
    "@dataclass",
    "class TrainConfig:",
    "    data_path:  str = '" + DATASET_PATH + "'",
    "    output_dir: str = '" + OUTPUT_DIR + "'",
    "    log_dir:    str = '" + LOG_DIR + "'",
    "    model_name: str = 'google-t5/t5-large'",
    "    task: str = 'question_generation'",
    "    max_input_len:  int = 128",   # was 256 — halved to save VRAM
    "    max_target_len: int = 128",   # was 256 — halved to save VRAM
    "    num_epochs:                  int   = 10",
    "    train_batch_size:            int   = 1",
    "    val_batch_size:              int   = 1",  # was 2
    "    gradient_accumulation_steps: int   = 32", # was 16, doubled to keep eff batch=32
    "    learning_rate:               float = 3e-5",
    "    weight_decay:                float = 0.01",
    "    warmup_ratio:                float = 0.15",
    "    max_grad_norm:               float = 0.3",
    "    use_fp16: bool = True",       # re-enable fp16 — cuts VRAM nearly in half
    "    eval_every_n_steps: int   = 200",
    "    save_every_n_steps: int   = 500",
    "    val_ratio:          float = 0.1",
    "    patience:           int   = 5",
    "    num_beams:            int   = 4",
    "    max_gen_len:          int   = 128",
    "    no_repeat_ngram_size: int   = 3",
    "    repetition_penalty:   float = 1.5",
    "    seed:                   int           = 42",
    "    num_workers:            int           = 0",  # was 2 — avoid worker memory overhead
    "    resume_from_checkpoint: Optional[str] = None",
    "",
    "CONFIG = TrainConfig()"
]

with open('/content/config.py', 'w') as f:
    f.write('\n'.join(config_lines))
print('config.py written')


config.py written


In [84]:
# Patch train.py to skip NaN batches instead of dying silently
src = open('/content/train.py').read()

old = (
    "            epoch_loss  += outputs.loss.item()\n"
    "            epoch_steps += 1"
)
new = (
    "            if torch.isnan(outputs.loss):\n"
    "                logger.warning('NaN loss detected at step ' + str(step) + ', skipping batch')\n"
    "                optimizer.zero_grad()\n"
    "                continue\n"
    "            epoch_loss  += outputs.loss.item()\n"
    "            epoch_steps += 1"
)

src = src.replace(old, new)
with open('/content/train.py', 'w') as f:
    f.write(src)

print('NaN guard patched into train.py')

NaN guard patched into train.py




```
# This is formatted as code
```

## Write `data_utils.py`

In [85]:
data_utils_code = '''
import json
import re
import random
from pathlib import Path
from typing import List, Tuple, Dict, Any

import torch
from torch.utils.data import Dataset
from transformers import T5Tokenizer


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\\s+", " ", text)
    text = re.sub(r"[^\\w\\s\\.\\,\\?\\!\\-\\(\\):]", "", text)
    return text.strip()


def load_dataset_from_path(data_path: str) -> List[Dict[str, Any]]:
    records = []
    path = Path(data_path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")
    if path.suffix == ".jsonl":
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
    elif path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            records = data if isinstance(data, list) else [data]
    else:
        raise ValueError(f"Unsupported file format: {path.suffix}")
    return records


def records_to_pairs(
    records: List[Dict[str, Any]],
    task: str = "question_generation"
) -> List[Tuple[str, str]]:
    pairs = []
    for record in records:
        question_text = record.get("question_text", "").strip()
        if not question_text:
            continue
        if task == "question_generation":
            course   = record.get("course_name", "Unknown")
            dept     = (record.get("departments") or ["Unknown"])[0]
            semester = record.get("semester", "Unknown")
            exam     = record.get("exam_type", "Unknown")
            co       = record.get("co", "")
            inp = f"course: {course} | department: {dept} | semester: {semester} | exam: {exam}"
            if co:
                inp += f" | CO: {co}"
            pairs.append((inp, question_text))
        elif task == "question_prediction":
            tgt = record.get("co", "") or record.get("course_name", "General")
            if tgt:
                pairs.append((question_text, tgt))
    return pairs


def split_pairs(
    pairs: List[Tuple[str, str]],
    val_ratio: float = 0.1,
    seed: int = 42
) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]]]:
    random.seed(seed)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * (1 - val_ratio))
    return pairs[:split_idx], pairs[split_idx:]


class ExamQuestionsDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input_len=256, max_target_len=256):
        self.pairs         = pairs
        self.tokenizer     = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        inp_text, tgt_text = self.pairs[idx]
        inp_enc = self.tokenizer(
            inp_text, max_length=self.max_input_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        tgt_enc = self.tokenizer(
            tgt_text, max_length=self.max_target_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        labels = tgt_enc["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      inp_enc["input_ids"].squeeze(),
            "attention_mask": inp_enc["attention_mask"].squeeze(),
            "labels":         labels.squeeze(),
        }


def build_datasets(data_path, tokenizer, task="question_generation",
                   max_input_len=256, max_target_len=256, val_ratio=0.1):
    records = load_dataset_from_path(data_path)
    pairs   = records_to_pairs(records, task=task)
    train_pairs, val_pairs = split_pairs(pairs, val_ratio=val_ratio)
    train_ds = ExamQuestionsDataset(train_pairs, tokenizer, max_input_len, max_target_len)
    val_ds   = ExamQuestionsDataset(val_pairs,   tokenizer, max_input_len, max_target_len)
    return train_ds, val_ds
'''

with open('/content/data_utils.py', 'w') as f:
    f.write(data_utils_code)

print('✅ data_utils.py written')

✅ data_utils.py written


## Write `train.py`

In [105]:
src = []
src.append("import sys, random, math, logging")
src.append("from pathlib import Path")
src.append("import numpy as np")
src.append("import torch")
src.append("from torch.utils.data import DataLoader")
src.append("from torch.optim import AdamW")
src.append("from torch.utils.tensorboard import SummaryWriter")
src.append("from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup")
src.append("from tqdm import tqdm")
src.append("from config import CONFIG, TrainConfig")
src.append("from data_utils import build_datasets")
src.append("")
src.append("logging.basicConfig(format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S', level=logging.INFO, handlers=[logging.StreamHandler(sys.stdout)])")
src.append("logger = logging.getLogger(__name__)")
src.append("")
src.append("def set_seed(seed):")
src.append("    random.seed(seed); np.random.seed(seed)")
src.append("    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)")
src.append("")
src.append("def count_parameters(model):")
src.append("    total = sum(p.numel() for p in model.parameters())")
src.append("    return 'Total: ' + str(round(total/1e6, 1)) + 'M'")
src.append("")
src.append("def save_checkpoint(model, tokenizer, optimizer, scheduler, step, loss, cfg):")
src.append("    ckpt_dir = Path(cfg.output_dir) / ('checkpoint-' + str(step))")
src.append("    ckpt_dir.mkdir(parents=True, exist_ok=True)")
src.append("    model.save_pretrained(ckpt_dir)")
src.append("    tokenizer.save_pretrained(ckpt_dir)")
src.append("    torch.save({'step': step, 'loss': loss, 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict()}, ckpt_dir / 'trainer_state.pt')")
src.append("    logger.info('Checkpoint saved -> ' + str(ckpt_dir))")
src.append("")
src.append("@torch.no_grad()")
src.append("def evaluate(model, val_loader, device, use_fp16):")
src.append("    model.eval()")
src.append("    total_loss, steps = 0.0, 0")
src.append("    for batch in val_loader:")
src.append("        input_ids      = batch['input_ids'].to(device)")
src.append("        attention_mask = batch['attention_mask'].to(device)")
src.append("        labels         = batch['labels'].to(device)")
src.append("        if use_fp16:")
src.append("            with torch.cuda.amp.autocast():")
src.append("                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)")
src.append("        else:")
src.append("            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)")
src.append("        if not torch.isnan(outputs.loss):")
src.append("            total_loss += outputs.loss.item(); steps += 1")
src.append("    model.train()")
src.append("    return total_loss / max(steps, 1)")
src.append("")
src.append("def train(cfg):")
src.append("    set_seed(cfg.seed)")
src.append("    SEP = '=' * 60")
src.append("    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')")
src.append("    if torch.cuda.is_available():")
src.append("        logger.info('GPU: ' + torch.cuda.get_device_name(0))")
src.append("        logger.info('Free VRAM: ' + str(round(torch.cuda.mem_get_info()[0]/1e9, 2)) + ' GB')")
src.append("    logger.info('Loading model: ' + cfg.model_name)")
src.append("    tokenizer = T5Tokenizer.from_pretrained(cfg.model_name)")
src.append("    model     = T5ForConditionalGeneration.from_pretrained(cfg.model_name)")
src.append("    model.gradient_checkpointing_enable()")  # saves ~30% VRAM
src.append("    model.to(device)")
src.append("    logger.info('Parameters -> ' + count_parameters(model))")
src.append("    if torch.cuda.is_available():")
src.append("        logger.info('VRAM after load: ' + str(round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.mem_get_info()[0])/1e9, 2)) + ' GB used')")
src.append("    train_ds, val_ds = build_datasets(cfg.data_path, tokenizer, cfg.task, cfg.max_input_len, cfg.max_target_len, cfg.val_ratio)")
src.append("    train_loader = DataLoader(train_ds, batch_size=cfg.train_batch_size, shuffle=True,  num_workers=cfg.num_workers, pin_memory=True)")
src.append("    val_loader   = DataLoader(val_ds,   batch_size=cfg.val_batch_size,   shuffle=False, num_workers=cfg.num_workers, pin_memory=True)")
src.append("    optimizer    = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)")
src.append("    total_steps  = math.ceil(len(train_loader) / cfg.gradient_accumulation_steps) * cfg.num_epochs")
src.append("    warmup_steps = int(total_steps * cfg.warmup_ratio)")
src.append("    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)")
src.append("    scaler = torch.amp.GradScaler('cuda') if (cfg.use_fp16 and device.type == 'cuda') else None")
src.append("    if scaler: logger.info('fp16 enabled')")
src.append("    Path(cfg.log_dir).mkdir(parents=True, exist_ok=True)")
src.append("    writer = SummaryWriter(log_dir=cfg.log_dir)")
src.append("    global_step, best_val_loss, no_improve = 0, float('inf'), 0")
src.append("    if cfg.resume_from_checkpoint and Path(cfg.resume_from_checkpoint).exists():")
src.append("        logger.info('Resuming from: ' + cfg.resume_from_checkpoint)")
src.append("        model = T5ForConditionalGeneration.from_pretrained(cfg.resume_from_checkpoint).to(device)")
src.append("        model.gradient_checkpointing_enable()")
src.append("        state = torch.load(Path(cfg.resume_from_checkpoint) / 'trainer_state.pt', map_location='cpu')")
src.append("        optimizer.load_state_dict(state['optimizer'])")
src.append("        scheduler.load_state_dict(state['scheduler'])")
src.append("        global_step, best_val_loss = state['step'], state['loss']")
src.append("    logger.info(SEP)")
src.append("    logger.info('  Task:       ' + cfg.task)")
src.append("    logger.info('  Epochs:     ' + str(cfg.num_epochs))")
src.append("    logger.info('  Train size: ' + str(len(train_ds)))")
src.append("    logger.info('  Val size:   ' + str(len(val_ds)))")
src.append("    logger.info('  Seq len:    ' + str(cfg.max_input_len))")
src.append("    logger.info('  Eff batch:  ' + str(cfg.train_batch_size * cfg.gradient_accumulation_steps))")
src.append("    logger.info(SEP)")
src.append("    model.train()")
src.append("    optimizer.zero_grad()")
src.append("    for epoch in range(1, cfg.num_epochs + 1):")
src.append("        epoch_loss, epoch_steps = 0.0, 0")
src.append("        progress = tqdm(train_loader, desc='Epoch ' + str(epoch) + '/' + str(cfg.num_epochs), leave=True)")
src.append("        for step, batch in enumerate(progress, 1):")
src.append("            input_ids      = batch['input_ids'].to(device)")
src.append("            attention_mask = batch['attention_mask'].to(device)")
src.append("            labels         = batch['labels'].to(device)")
src.append("            try:")
src.append("                if scaler:")
src.append("                    with torch.amp.autocast('cuda'):")
src.append("                        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)")
src.append("                    loss = outputs.loss / cfg.gradient_accumulation_steps")
src.append("                    scaler.scale(loss).backward()")
src.append("                else:")
src.append("                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)")
src.append("                    loss = outputs.loss / cfg.gradient_accumulation_steps")
src.append("                    loss.backward()")
src.append("            except RuntimeError as e:")
src.append("                if 'out of memory' in str(e):")
src.append("                    logger.warning('OOM at step ' + str(step) + ', skipping batch')")
src.append("                    optimizer.zero_grad()")
src.append("                    torch.cuda.empty_cache()")
src.append("                    continue")
src.append("                raise e")
src.append("            if torch.isnan(outputs.loss):")
src.append("                logger.warning('NaN at step ' + str(step) + ', skipping')")
src.append("                optimizer.zero_grad(); continue")
src.append("            epoch_loss  += outputs.loss.item()")
src.append("            epoch_steps += 1")
src.append("            if step % cfg.gradient_accumulation_steps == 0:")
src.append("                if scaler: scaler.unscale_(optimizer)")
src.append("                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)")
src.append("                if scaler:")
src.append("                    scaler.step(optimizer); scaler.update()")
src.append("                else:")
src.append("                    optimizer.step()")
src.append("                scheduler.step()")
src.append("                optimizer.zero_grad()")
src.append("                global_step += 1")
src.append("                writer.add_scalar('Loss/train_step', outputs.loss.item(), global_step)")
src.append("                writer.add_scalar('LR', scheduler.get_last_lr()[0], global_step)")
src.append("                progress.set_postfix(loss=round(outputs.loss.item(), 4), lr=round(scheduler.get_last_lr()[0], 7))")
src.append("                if global_step % cfg.eval_every_n_steps == 0:")
src.append("                    val_loss = evaluate(model, val_loader, device, cfg.use_fp16)")
src.append("                    writer.add_scalar('Loss/val', val_loss, global_step)")
src.append("                    logger.info('[Step ' + str(global_step) + '] Val: ' + str(round(val_loss, 4)) + ' | Best: ' + str(round(best_val_loss, 4)))")
src.append("                    if val_loss < best_val_loss:")
src.append("                        best_val_loss, no_improve = val_loss, 0")
src.append("                        best_dir = Path(cfg.output_dir) / 'best_model'")
src.append("                        best_dir.mkdir(parents=True, exist_ok=True)")
src.append("                        model.save_pretrained(best_dir)")
src.append("                        tokenizer.save_pretrained(best_dir)")
src.append("                        logger.info('Best model saved -> ' + str(best_dir))")
src.append("                    else:")
src.append("                        no_improve += 1")
src.append("                        if no_improve >= cfg.patience:")
src.append("                            logger.info('Early stopping triggered.')")
src.append("                            final_dir = Path(cfg.output_dir) / 'final_model'")
src.append("                            final_dir.mkdir(parents=True, exist_ok=True)")
src.append("                            model.save_pretrained(final_dir)")
src.append("                            tokenizer.save_pretrained(final_dir)")
src.append("                            writer.close(); return")
src.append("                if global_step % cfg.save_every_n_steps == 0:")
src.append("                    save_checkpoint(model, tokenizer, optimizer, scheduler, global_step, epoch_loss / max(epoch_steps, 1), cfg)")
src.append("        avg = epoch_loss / max(epoch_steps, 1)")
src.append("        writer.add_scalar('Loss/train_epoch', avg, epoch)")
src.append("        logger.info('Epoch ' + str(epoch) + ' done | Avg Loss: ' + str(round(avg, 4)))")
src.append("    final_dir = Path(cfg.output_dir) / 'final_model'")
src.append("    final_dir.mkdir(parents=True, exist_ok=True)")
src.append("    model.save_pretrained(final_dir)")
src.append("    tokenizer.save_pretrained(final_dir)")
src.append("    logger.info('Training complete -> ' + str(final_dir))")
src.append("    logger.info('Best val loss: ' + str(round(best_val_loss, 4)))")
src.append("    writer.close()")
src.append("")
src.append("if __name__ == '__main__':")
src.append("    train(CONFIG)")

with open('/content/train.py', 'w') as f:
    f.write('\n'.join(src))
print('train.py written')

train.py written


##  Write `inference.py`

In [95]:
inference_code = '''
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from config import CONFIG


def load_model(model_path: str):
    print(f"Loading model from: {model_path}")
    tokenizer = T5Tokenizer.from_pretrained(model_path)
    model     = T5ForConditionalGeneration.from_pretrained(model_path)
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    print(f"Model ready on {device}")
    return model, tokenizer, device


def generate(model, tokenizer, device, input_text, num_beams=4,
             max_length=256, num_return_sequences=1,
             no_repeat_ngram=3, repetition_penalty=1.5):
    inputs = tokenizer(input_text, return_tensors="pt",
                       max_length=256, truncation=True, padding=True).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=no_repeat_ngram,
            repetition_penalty=repetition_penalty,
            early_stopping=True,
        )
    return [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]


def build_generation_prompt(course, department="", semester="EVEN SEMESTER",
                             exam="T-3", outcome="general"):
    return (f"course: {course} | department: {department} | "
            f"semester: {semester} | exam: {exam} | CO: {outcome}")


def generate_paper(model, tokenizer, device, course, department,
                   outcomes, num_questions_per_outcome=2, cfg=CONFIG):
    paper = {}
    for co in outcomes:
        prompt = build_generation_prompt(course, department, outcome=co)
        paper[co] = generate(
            model, tokenizer, device, prompt,
            num_beams=cfg.num_beams, max_length=cfg.max_gen_len,
            num_return_sequences=num_questions_per_outcome,
            no_repeat_ngram=cfg.no_repeat_ngram_size,
            repetition_penalty=cfg.repetition_penalty,
        )
    return paper


def print_paper(paper, course):
    print(f"\\n{\' \' *60}")
    print(f"  Generated Question Paper — {course}")
    print("=" * 60)
    q_num = 1
    for co, questions in paper.items():
        print(f"\\n[{co}]")
        for q in questions:
            print(f"  Q{q_num}. {q}"); q_num += 1
    print("=" * 60)
'''

with open('/content/inference.py', 'w') as f:
    f.write(inference_code)

print('✅ inference.py written')

✅ inference.py written


##  Inspect Dataset (prepare_data)

In [96]:
import sys
sys.path.insert(0, '/content')

import json
from collections import Counter
from data_utils import load_dataset_from_path, records_to_pairs, split_pairs, clean_text
from config import CONFIG


def analyze_and_print(data_path, task="question_generation", val_ratio=0.1):
    records = load_dataset_from_path(data_path)

    stats = {
        "total": len(records), "empty": 0, "has_marks": 0, "has_co": 0,
        "courses": Counter(), "departments": Counter(),
        "exam_types": Counter(), "semesters": Counter(), "cos": Counter(),
    }
    total_len = 0

    for r in records:
        q = r.get("question_text", "")
        if not q.strip(): stats["empty"] += 1; continue
        total_len += len(clean_text(q).split())
        stats["courses"][r.get("course_name", "Unknown")] += 1
        for dept in (r.get("departments") or ["Unknown"]):
            stats["departments"][dept] += 1
        stats["exam_types"][r.get("exam_type", "Unknown")] += 1
        stats["semesters"][r.get("semester", "Unknown")] += 1
        co = r.get("co", "")
        stats["cos"][co if co else "none"] += 1
        if r.get("marks"): stats["has_marks"] += 1
        if co: stats["has_co"] += 1

    valid = stats["total"] - stats["empty"]
    avg_len = round(total_len / max(valid, 1), 1)

    print("\n" + "="*60)
    print("  DATASET ANALYSIS")
    print("="*60)
    print(f"  Total records   : {stats['total']}")
    print(f"  Empty questions : {stats['empty']}")
    print(f"  Valid records   : {valid}")
    print(f"  With marks      : {stats['has_marks']}")
    print(f"  With CO         : {stats['has_co']}")
    print(f"  Avg question len: {avg_len} words")
    print(f"\n  Top 10 Courses:")
    for name, cnt in stats["courses"].most_common(10):
        print(f"    {cnt:4d}  {name[:55]}")
    print(f"\n  Departments : {dict(stats['departments'].most_common())}")
    print(f"  Exam types  : {dict(stats['exam_types'].most_common())}")
    print(f"  Semesters   : {dict(stats['semesters'].most_common())}")
    print(f"  Top COs     : {dict(stats['cos'].most_common(10))}")
    print("="*60)

    pairs = records_to_pairs(records, task=task)
    short = sum(1 for _, tgt in pairs if len(tgt.split()) < 5)
    long  = sum(1 for _, tgt in pairs if len(tgt.split()) > 200)
    print(f"\n  Total pairs : {len(pairs)}")
    print(f"  Short (<5w) : {short} | Long (>200w): {long}")

    train_pairs, val_pairs = split_pairs(pairs, val_ratio=val_ratio)
    print(f"  Train split : {len(train_pairs)} | Val split: {len(val_pairs)}")
    print("\n  Sample input:")
    for inp, tgt in train_pairs[:3]:
        print(f"    IN : {inp}")
        print(f"    OUT: {tgt[:120]}...\n")
    print("\n✅ Dataset looks good. Proceed to training.")


analyze_and_print(CONFIG.data_path, task=CONFIG.task, val_ratio=CONFIG.val_ratio)


  DATASET ANALYSIS
  Total records   : 8099
  Empty questions : 1
  Valid records   : 8098
  With marks      : 2350
  With CO         : 1369
  Avg question len: 51.1 words

  Top 10 Courses:
     241  
      96  Biochemistry
      84  Foundation For Data Science And Visualization
      60  Introduction To Bioinformatics
      60  Transportation Engineering
      60  Immunology
      53  Computer Aided Drug Design
      53  Downstream Processing
      50  Artificial Intelligence Techniques
      50  Highway Engineering

  Departments : {'BT': 1991, 'IT': 1804, 'CSE': 1754, 'ECE': 1748, 'Civil': 1602, 'CREDITS': 1437, 'BI': 1034, 'Physics': 141, 'Microbiology': 123, 'Unknown': 120, 'All Branches': 116, 'CS': 110, 'ECM': 82, 'ALL': 54, 'Math': 50, 'Structural Engineering': 46, 'Credits': 45, 'PMS': 41, 'Open Elective': 40, 'Mathematics and Computing': 37, 'VLSI': 35, 'Biotechnology': 35, 'Mathematics': 34, 'Data Science': 34, 'CSE-AIML': 21, 'CSE-AIDS': 21, 'CSE-CS': 21, 'HSS': 20, 'BBA'

## Training
> This cell runs the full training loop. Monitor the tqdm progress bar for live loss.  
> Checkpoints + best model are saved to your Google Drive automatically.

In [ ]:
import sys, gc, torch

# Free everything
try: del model
except: pass
try: del tokenizer
except: pass
gc.collect()
torch.cuda.empty_cache()
print('Free VRAM:', round(torch.cuda.mem_get_info()[0]/1e9, 2), 'GB')

# Rewrite config — fp16 OFF, everything else same
config_lines = [
    "from dataclasses import dataclass",
    "from typing import Optional",
    "",
    "@dataclass",
    "class TrainConfig:",
    "    data_path:  str = '" + DATASET_PATH + "'",
    "    output_dir: str = '" + OUTPUT_DIR + "'",
    "    log_dir:    str = '" + LOG_DIR + "'",
    "    model_name: str = 'google-t5/t5-base'", # Changed to t5-base to save VRAM
    "    task: str = 'question_generation'",
    "    max_input_len:  int = 64",   # Further reduced to save VRAM
    "    max_target_len: int = 64",   # Further reduced to save VRAM
    "    num_epochs:                  int   = 10",
    "    train_batch_size:            int   = 1",
    "    val_batch_size:              int   = 1",
    "    gradient_accumulation_steps: int   = 16", # Halved again for VRAM
    "    learning_rate:               float = 1e-4",
    "    weight_decay:                float = 0.01",
    "    warmup_ratio:                float = 0.15",
    "    max_grad_norm:               float = 1.0",
    "    use_fp16: bool = False",      # THE FIX — fp16 causes NaN on T5+grad_ckpt
    "    eval_every_n_steps: int   = 200",
    "    save_every_n_steps: int   = 500",
    "    val_ratio:          float = 0.1",
    "    patience:           int   = 5",
    "    num_beams:            int   = 4",
    "    max_gen_len:          int   = 128",
    "    no_repeat_ngram_size: int   = 3",
    "    repetition_penalty:   float = 1.5",
    "    seed:                   int           = 42",
    "    num_workers:            int           = 0",
    "    resume_from_checkpoint: Optional[str] = None",
    "",
    "CONFIG = TrainConfig()"
]

with open('/content/config.py', 'w') as f:
    f.write('\n'.join(config_lines))
print('config.py written — fp16 disabled, sequence lengths and accumulation steps reduced, model changed to t5-base')

# Reload and train
for mod in ['config', 'train', 'data_utils']:
    if mod in sys.modules:
        del sys.modules[mod]

from config import CONFIG
from train import train
train(CONFIG)

Free VRAM: 15.43 GB
config.py written — fp16 disabled, sequence lengths and accumulation steps reduced, model changed to t5-base


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:  45%|####4     | 398M/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch 1/10:  45%|████▍     | 3276/7288 [09:49<13:42,  4.88it/s, loss=4.81, lr=2.98e-5]

In [ ]:
# Inspect train.py around line 117 to verify the fix
with open('/content/train.py', 'r') as f:
    lines = f.readlines()
    # Showing lines 110 to 125
    for i, line in enumerate(lines[110:125], 111):
        print(f"{i}: {line}", end='')

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## Inference: Generate Questions

In [ ]:
import sys
sys.path.insert(0, '/content')

from config import CONFIG
from inference import load_model, generate, build_generation_prompt, generate_paper, print_paper

MODEL_PATH = f"{OUTPUT_DIR}/best_model"
model, tokenizer, device = load_model(MODEL_PATH)

In [ ]:
# ── Single question generation ─────────────────────────────────────────
prompt = build_generation_prompt(
    course="Data Structures and Algorithms",
    department="CSE",
    semester="ODD SEMESTER",
    exam="T-2",
    outcome="CO3"
)
print(f"Prompt: {prompt}\n")

results = generate(
    model, tokenizer, device, prompt,
    num_beams=CONFIG.num_beams,
    max_length=CONFIG.max_gen_len,
    num_return_sequences=3,
    no_repeat_ngram=CONFIG.no_repeat_ngram_size,
    repetition_penalty=CONFIG.repetition_penalty,
)

print("Generated questions:")
for i, q in enumerate(results, 1):
    print(f"  Q{i}. {q}")

In [ ]:
# ── Generate full mini question paper ──────────────────────────────────
paper = generate_paper(
    model, tokenizer, device,
    course="Computer Networks",
    department="CSE",
    outcomes=["CO1", "CO2", "CO3", "CO4", "CO5"],
    num_questions_per_outcome=2,
    cfg=CONFIG,
)
print_paper(paper, course="Computer Networks")

## Cell 12 — Resume Interrupted Training
> If Colab disconnected mid-training, run this cell to pick up from the last checkpoint.

In [ ]:
import os, sys
sys.path.insert(0, '/content')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Re-write source files (run cells 4-7 first, or uncomment below)
# exec(open('/content/config.py').read())  # already written if you ran cell 4

# Find latest checkpoint
import glob
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
               key=lambda p: int(p.split('-')[-1]))
if ckpts:
    latest = ckpts[-1]
    print(f"Latest checkpoint: {latest}")
    CONFIG.resume_from_checkpoint = latest
else:
    print("No checkpoints found — starting fresh")

from train import train
train(CONFIG)